load resuslts of channel-wise power perturbation for each subject.
make separate plot for each channel and freq-band in beginning.
do TSNE UMAP ofer all trials with trials colored by subject color

perhaps also use time-point groups for clustering results


In [ ]:
import pathlib
import random
import copy
import numpy as np
import torch

from captum.attr import *
import matplotlib.pyplot as plt
from omegaconf import OmegaConf
import yaml
import argparse
from data.data_loader import load_eeg_data, get_sliding_window_data, create_dataloader
import os
import mne
from tqdm import tqdm
from sklearn.preprocessing import MinMaxScaler
import pandas as pd

from scipy.cluster.hierarchy import fcluster

from models.s4net import S4PatchedFinalNet,TrunkNet, HeadNet

In [ ]:
CFG_YAML = """
wandb:
 key: f0c92a0059bf12e2647f0a1c22fdcd12555fa6df
model:
dataset:
 data_directory: /home/marco/Documents/GitHub/tms_eeg_decoding/data
 #file_name: subject_{:03d}_preprocessed_combined_py.fif
 file_name: subject_{:03d}_preprocessed_combined_py.fif
 exclude_timepoints: 100
 subject_index: 1
 test_subject_indices: [1,2,13,24,26,27,29,34,35,41, 42,43,45,46,47,48,52,55,56,57,60,62,67,69,72,73,79,80,86,88,92,102]
 #test_subject_indices: [2]
training:
 training_start_len: 100
 pretrain_epochs: 100
 pretrain_lr: 0.0001
 val_window_len: 1
 epochs_per_window: 10
 num_warmup_epochs: 5
 num_epochs: 800
 slide_step: 1
 num_warmup_epochs_per_window: 0
 lr: 0.005 #maybe change back to 0.0001
 nll_beta: 0.001
 num_warmup_epochs: 0
 batch_size: 50 #better to use 50
 random_seed: 42
 precision: bf16
 kde_lambda: 0.5
 finetune_entire_model: true # Set to true to finetune the entire model, false for transformer only
exp_name: S4_S4EEGNet_ema
"""

def load_config():
    cfg = OmegaConf.create(yaml.safe_load(CFG_YAML))
    cfg.exp_name = f"{cfg.exp_name}_subject_{cfg.dataset.subject_index}"
    return cfg

def parse_args():
    parser = argparse.ArgumentParser()
    parser.add_argument("--update_conf", nargs="*", help="Updates to the configuration in the form of key=value pairs", default=[])
    parser.add_argument("-f", "--fff", help="A dummy argument to handle IPython's default argument", default="1")
    return parser.parse_args()

def update_config(cfg, cli_args):
    for update in cli_args.update_conf:
        key, value = update.split("=")
        try:
            value = eval(value)
        except:
            pass
        OmegaConf.update(cfg, key, value, force_add=True)
    cfg.exp_name = cfg.exp_name + "_" + "_".join(cli_args.update_conf)
    print(OmegaConf.to_yaml(cfg))
    return cfg


def save_config(cfg):
    os.makedirs("conf/sweeps", exist_ok=True)
    os.makedirs("exp/withinsubs", exist_ok=True)
    with open(f"conf/sweeps/withinsubs_{cfg.exp_name}.yaml", "w") as f:
        f.write(OmegaConf.to_yaml(cfg))

In [ ]:
import matplotlib.pylab as pylab
params = {'legend.fontsize': 'x-large',
          'figure.titlesize': 'x-large',
          'figure.figsize': (15, 5),
         'axes.labelsize': 'x-large',
         'axes.titlesize':'x-large',
         'xtick.labelsize':'x-large',
         'ytick.labelsize':'x-large'}
pylab.rcParams.update(params)

In [ ]:

def calculate_diff_per_channel(pred_label_original, freq_bands, amplification_factors, ch_names, subject_index=2):
    mean_diff_per_channel = {}
    median_diff_per_channel = {}

    for band_name, (low_freq, high_freq) in freq_bands.items():
        mean_diff_per_channel[band_name] = {}
        median_diff_per_channel[band_name] = {}
        for factor in amplification_factors:
            perturbed_data = np.load(f"perturbed_predictions/perturbed_prediction_dict_{band_name}_channel_amp_factor_{factor}_subject_{subject_index}.npy", allow_pickle=True).item()
            mean_diff_per_channel[band_name][factor] = {}
            median_diff_per_channel[band_name][factor] = {}
            for ch_name in ch_names:
                perturbed_amplitude = perturbed_data[ch_name][0]
                diff = np.abs(pred_label_original - perturbed_amplitude)
                mean_diff_per_channel[band_name][factor][ch_name] = np.mean(diff)
                median_diff_per_channel[band_name][factor][ch_name] = np.median(diff)
    
    return mean_diff_per_channel, median_diff_per_channel

In [ ]:
def get_top_channels(median_diff_per_channel, freq_bands, amplification_factors, top_k=10):
    top_channels_median = {}
    prediction_diff_median = {}
    
    for band_name in freq_bands.keys():
        top_channels_median[band_name] = {}
        prediction_diff_median[band_name] = {}
        
        for factor in amplification_factors:
            median_diffs = median_diff_per_channel[band_name][factor]
            
            # Sort the channels based on their differences
            sorted_median_diffs = sorted(median_diffs.items(), key=lambda item: item[1], reverse=True)
            
            # Select the top k channels
            top_channels_median[band_name][factor] = [ch for ch, _ in sorted_median_diffs[:top_k]]
            
            # Store the prediction differences for the top k channels
            prediction_diff_median[band_name][factor] = {ch: median_diffs[ch] for ch in top_channels_median[band_name][factor]}
    

    return top_channels_median,prediction_diff_median


In [ ]:
import pickle

def load_predicted_amplitude_for_subject(subject_index=2):
    data_dir = "/home/marco/Documents/GitHub/tms_eeg_decoding/data/explanation_data"
    file_path = os.path.join(data_dir, f"subject_{subject_index}_results.pkl")

    with open(file_path, 'rb') as f:
        subject_data = pickle.load(f)   
    predictions, uncertainties, explanations = subject_data['predictions'], subject_data['uncertainties'], subject_data['explanations']

        
    cfg = load_config()
    cfg.dataset.subject_index = subject_index
    _, _, _, _, _, _, _, ch_names = load_eeg_data(cfg)
    
    return predictions, uncertainties, explanations, ch_names

In [ ]:
def load_subject_topk(subject_index=2):
    topk = np.load("top_k_abs.npy", allow_pickle=True).item()
    return topk[subject_index]

In [ ]:
def get_top_k_keys(d, k):
    """
    Returns the top k keys in a dictionary that have the highest values.

    Parameters:
    d (dict): The input dictionary.
    k (int): The number of top keys to return.

    Returns:
    list: A list of the top k keys with the highest values.
    """
    # Sort the dictionary by values in descending order and get the top k keys
    top_k_keys = sorted(d, key=d.get, reverse=True)[:k]
    return top_k_keys

# Example usage
d = {'a': 10, 'b': 20, 'c': 15, 'd': 5, 'e': 25}
k = 3
print(get_top_k_keys(d, k))  # Output: ['e', 'b', 'c']

In [ ]:
freq_bands = {
              "theta": (0, 4),
              "delta": (4, 8),
              "alpha": (8, 12),
              "beta": (12, 30),
              "gamma": (30, 45)}
amplification_factors = [0.5, 0.75, 1.5,2 , 3 ]

In [ ]:
dir = "/home/marco/Documents/GitHub/tms_eeg_decoding/perturb_samples_power/perturbed_predictions"

In [ ]:
def calculate_agreement_top_channels(top_channels_median, top_channels_top_k_2, freq_bands, amplification_factors):
    """
    Calculate the agreement ratio of top channels between absolute prediction difference and interpretability method.

    Parameters:
    top_channels_median (dict): Dictionary containing top channels based on median differences.
    top_channels_top_k_2 (list): List of top channels from interpretability method.
    freq_bands (dict): Dictionary of frequency bands.
    amplification_factors (list): List of amplification factors.

    Returns:
    dict: Agreement results for each frequency band and amplification factor.
    """
    # Initialize a dictionary to store the agreement results
    agreement_results = {}

    # Iterate over each frequency band
    for band_name in freq_bands.keys():
        agreement_results[band_name] = {}
        for factor in amplification_factors:
            # Extract the top 10 channels from top_channels_median for the current band and factor
            top_channels_median_band_factor = top_channels_median[band_name][factor]
            
            # Check the number of top channels that agree
            agreement_count = sum(1 for ch in top_channels_top_k_2 if ch in top_channels_median_band_factor)
            
            # Calculate the agreement ratio
            agreement_ratio = agreement_count / 10.0
            
            # Store the result
            agreement_results[band_name][factor] = agreement_ratio

    return agreement_results

In [ ]:
import itertools

In [ ]:
cfg = load_config()
pairs = list(itertools.combinations(cfg.dataset.test_subject_indices, 2)) 

In [ ]:
def load_data_all_subjects():
    all_subjects_data = {}
    for subject_index in cfg.dataset.test_subject_indices:
        predictions, uncertainties, _, ch_names = load_predicted_amplitude_for_subject(subject_index)
        top_k = load_subject_topk(subject_index)
        

        file_path = "/home/marco/Documents/GitHub/tms_eeg_decoding/data/subject_{:03d}_preprocessed_combined_py.fif".format(subject_index)
        epochs = mne.read_epochs(file_path)
        info_subj = epochs.info
        all_subjects_data[subject_index] = {"predictions": predictions, "uncertainties": uncertainties, "top_k": top_k, "ch_names": ch_names, "info_subj": info_subj}
    return all_subjects_data

In [ ]:
all_subjects_data = load_data_all_subjects()

In [ ]:
 #   mean_diff_per_channel, median_diff_per_channel = calculate_diff_per_channel(original_predictions, freq_bands, amplification_factors, ch_names, #subject_index=subject_index)
def median_difference_all_subjects(data_all_subjects, take_abs=True):
    median_diff_per_channel_all_subjects = {}
    for subject_index, data in data_all_subjects.items():
        predictions = data["predictions"]
        ch_names = data["ch_names"]
        median_diff_per_channel = {}
        for band_name, (low_freq, high_freq) in freq_bands.items():
            median_diff_per_channel[band_name] = {}
            for factor in amplification_factors:
                perturbed_data = np.load(f"{dir}/perturbed_prediction_dict_{band_name}_channel_amp_factor_{factor}_subject_{subject_index}.npy", allow_pickle=True).item()
                median_diff_per_channel[band_name][factor] = {}
                for ch_name in ch_names:
                    perturbed_amplitude = perturbed_data[ch_name][0]
                    if take_abs:
                        diff = np.abs(predictions - perturbed_amplitude)
                    else:
                        diff = predictions - perturbed_amplitude
                    median_diff_per_channel[band_name][factor][ch_name] = np.median(diff)
        median_diff_per_channel_all_subjects[subject_index] = median_diff_per_channel
    return median_diff_per_channel_all_subjects


In [ ]:
median_diff_per_channel_all_subjects = median_difference_all_subjects(all_subjects_data)

In [ ]:
def get_top_channels_all_subjects(median_diff_per_channel_all_subjects):
    top_channels_median_all_subjects = {}
    prediction_diff_median_all_subjects = {}
    for subject_index, median_diff_per_channel in median_diff_per_channel_all_subjects.items():
        top_channels_median,prediction_diff_median = get_top_channels(median_diff_per_channel, freq_bands, amplification_factors)
        top_channels_median_all_subjects[subject_index] = top_channels_median
        prediction_diff_median_all_subjects[subject_index] = prediction_diff_median
    return top_channels_median_all_subjects, prediction_diff_median_all_subjects

In [ ]:
top_channels_median_all_subjects, prediction_diff_median_all_subjects = get_top_channels_all_subjects(median_diff_per_channel_all_subjects)

In [ ]:
def calculate_agreement_top_channels(top_channels_median_1, top_channels_median_2, freq_bands, amplification_factors):
    """
    Calculate the agreement ratio of top channels between two subjects.

    Parameters:
    top_channels_median_1 (dict): Dictionary containing top channels for subject 1.
    top_channels_median_2 (dict): Dictionary containing top channels for subject 2.
    freq_bands (dict): Dictionary of frequency bands.
    amplification_factors (list): List of amplification factors.

    Returns:
    dict: Agreement results for each frequency band and amplification factor.
    """
    agreement_results = {}

    for band_name in freq_bands.keys():
        agreement_results[band_name] = {}
        for factor in amplification_factors:
            # Extract the top 10 channels for both subjects
            top_channels_1 = top_channels_median_1[band_name][factor]
            top_channels_2 = top_channels_median_2[band_name][factor]
            
            # Calculate agreement
            agreement_count = sum(1 for ch in top_channels_1 if ch in top_channels_2)
            agreement_ratio = agreement_count / 10.0
            
            agreement_results[band_name][factor] = agreement_ratio

    return agreement_results


In [ ]:
def plot_agreement_matrix(agreement_results, subject_index1, subject_index2, ax=None, subfig=None):
    """
    Plot the agreement matrix.

    Parameters:
    agreement_results (dict): Agreement results for each frequency band and amplification factor.
    subject_index1 (int): Index of first subject
    subject_index2 (int): Index of second subject
    ax (matplotlib.axes.Axes, optional): Axis to plot on. If None and subfig is None, creates new figure.
    subfig (matplotlib.figure.SubFigure, optional): Subfigure to plot on.
    """
    # Prepare data for plotting
    bands = list(agreement_results.keys())
    factors = list(agreement_results[bands[0]].keys())
    agreement_matrix = np.zeros((len(bands), len(factors)))

    for i, band in enumerate(bands):
        for j, factor in enumerate(factors):
            agreement_matrix[i, j] = agreement_results[band][factor]

    # Create new figure/axis based on input
    if subfig is not None:
        ax = subfig.add_subplot(111)
        fig = subfig.figure
    elif ax is None:
        fig, ax = plt.subplots(figsize=(5, 5))
    else:
        fig = ax.figure

    # Plot the agreement matrix
    cax = ax.matshow(agreement_matrix, cmap='viridis', vmin=0, vmax=1)

    # Add color bar
    fig.colorbar(cax, ticks=np.arange(0, 1.1, 0.2), ax=ax).set_label('Agreement Ratio')

    # Set axis labels
    ax.set_xticks(np.arange(len(factors)))
    ax.set_yticks(np.arange(len(bands)))
    ax.set_xticklabels(factors)
    ax.set_yticklabels(bands)
    ax.set_title(f'Agreement between subject {subject_index1} and subject {subject_index2}')

    # Rotate the tick labels and set their alignment
    plt.setp(ax.get_xticklabels(), rotation=45, ha="left", rotation_mode="anchor")

    # Add text annotations
    for i in range(len(bands)):
        for j in range(len(factors)):
            text = ax.text(j, i, f"{agreement_matrix[i, j]:.2f}", ha="center", va="center", color="white")

    ax.set_xlabel('Amplification factors')
    ax.set_ylabel('Frequency Bands')

    return ax

In [ ]:
def plot_agreement_topomap(top_channels_median1, top_channels_median2, amp_factor, ch_names, info_subj1, freq_bands, subfig=None):
    # Create figure based on input
    if subfig is not None:
        fig = subfig
    else:
        fig, axs = plt.subplots(1, len(freq_bands.keys()), figsize=(20, 4))

    # Get axes based on figure type
    if subfig is not None:
        axs = fig.subplots(1, len(freq_bands.keys()))

    common_channels = {}
    for ax, band_name in zip(axs, freq_bands.keys()):
        common_channels[band_name] = np.intersect1d(top_channels_median1[band_name][amp_factor], 
                                                  top_channels_median2[band_name][amp_factor])
        agree_channels = np.zeros(len(ch_names))
        for ch in common_channels[band_name]:
            agree_channels[ch_names.index(ch)] = 1

        mne.viz.plot_topomap(agree_channels, info_subj1, show=False, names=ch_names, 
                           axes=ax, image_interp="nearest")
        ax.set_title(f'{band_name} band')

    return axs


# pairwise agreement

In [ ]:
cfg = load_config()
for subject_index1, subject_index2 in itertools.combinations(cfg.dataset.test_subject_indices,2):
    fig = plt.figure(layout='constrained', figsize=(15, 4))
    subfigs = fig.subfigures(1, 2, wspace=0.07)
    agreement_results = calculate_agreement_top_channels(top_channels_median_all_subjects[subject_index1], top_channels_median_all_subjects[subject_index2], freq_bands, amplification_factors)
    plot_agreement_matrix(agreement_results, subject_index1, subject_index2, subfig=subfigs[0])
    plot_agreement_topomap(top_channels_median_all_subjects[subject_index1], top_channels_median_all_subjects[subject_index2], 2, all_subjects_data[subject_index1]["ch_names"], all_subjects_data[subject_index1]["info_subj"], freq_bands, subfig=subfigs[1])




In [ ]:
agreement_results

notably neighboring channels of C3 are often in the most important channels, especially for the lower frequency bands.


when clustering, potentially use a weighting that takes into account the spatial distance of channel to account for spatial imprecision on EEG.

make a single matrix plot showing subject-pair agreement per frequency band for a given amplification factor

ch_names

# clustering

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.cluster.hierarchy import linkage, dendrogram
from scipy.spatial.distance import pdist, squareform

def spatial_channel_similarity(top_channels_i, top_channels_j, common_channels, 
                              channel_dist_matrix_i, channel_dist_matrix_j, sigma=0.1):
    """
    Compute similarity between two subjects' top channels accounting for spatial proximity,
    using subject-specific distance matrices.
    
    Parameters:
    -----------
    top_channels_i, top_channels_j : list
        Lists of channel names for the top channels of subjects i and j
    ch_names : list
        List of all channel names
    channel_dist_matrix_i : np.array
        Distance matrix for subject i
    channel_dist_matrix_j : np.array
        Distance matrix for subject j
    sigma : float
        Parameter controlling how quickly similarity falls off with distance
    
    Returns:
    --------
    float : Similarity score between 0 and 1
    """
    # Convert channel names to indices
    # Right now this computes the channel index in common channels for each of the top 10 channels of a subject
    indices_i = [common_channels.index(ch) for ch in top_channels_i if ch in common_channels]
    indices_j = [common_channels.index(ch) for ch in top_channels_j if ch in common_channels]


    # Using subject i's distance matrix
    # For each channel in subject i, find the closest match in subject j of the top 10 channelss
    similarities_i_to_j = []
    # for each channel in the top 10 of subect 1, search for the channel in the top 10 of subject 2 that is most similar weighted by a gaussian kernel
    # computes distance to each channel in the top 10 of subject 2
    for idx_i in indices_i:
        dists = [channel_dist_matrix_i[idx_i, idx_j] for idx_j in indices_j]
        similarities = [np.exp(-d**2/sigma) for d in dists]
        # find the most similar channel
        similarities_i_to_j.append(max(similarities))
    
    # Using subject j's distance matrix
    # For each channel in subject j, find the closest match in subject i
    similarities_j_to_i = []
    for idx_j in indices_j:
        dists = [channel_dist_matrix_j[idx_j, idx_i] for idx_i in indices_i]
        # does a gaussian kernel make sense here?
        similarities = [np.exp(-d**2/sigma) for d in dists]
        similarities_j_to_i.append(max(similarities))
    
    # Combine the similarities (average)
    overall_similarity = (sum(similarities_i_to_j) + sum(similarities_j_to_i)) / (len(indices_i) + len(indices_j))
    
    return overall_similarity

def compute_similarity_matrix(top_channels_by_subject, common_channels, channel_dist_matrices, sigma=0.1):
    """
    Compute similarity matrix between all pairs of subjects using subject-specific distance matrices.
    
    Parameters:
    -----------
    top_channels_by_subject : dict
        Dictionary with subject IDs as keys and lists of top channel names as values
    ch_names : list
        List of all channel names
    channel_dist_matrices : dict
        Dictionary with subject IDs as keys and channel distance matrices as values
    sigma : float
        Parameter controlling how quickly similarity falls off with distance
    
    Returns:
    --------
    sim_matrix : np.array
        Similarity matrix
    subjects : list
        List of subject IDs
    """
    subjects = list(top_channels_by_subject.keys())
    n_subjects = len(subjects)
    
    # Initialize similarity matrix
    sim_matrix = np.zeros((n_subjects, n_subjects))
    
    # Compute pairwise similarities
    for i in range(n_subjects):
        subj_i = subjects[i]
        for j in range(i, n_subjects):
            subj_j = subjects[j]
            
            if i == j:
                # self similarity should be highest and is set to 1 here
                sim_matrix[i, j] = 1.0  
            else:
                # Use each subject's own distance matrix
                sim = spatial_channel_similarity(
                    top_channels_by_subject[subj_i],
                    top_channels_by_subject[subj_j],
                    common_channels,
                    channel_dist_matrices[subj_i],
                    channel_dist_matrices[subj_j],
                    sigma
                )
                sim_matrix[i, j] = sim
                sim_matrix[j, i] = sim  
    
    return sim_matrix, subjects

def visualize_similarity_matrix(sim_matrix, subjects):
    """
    Visualize the similarity matrix as a heatmap.
    """
    plt.figure(figsize=(12, 10))
    plt.imshow(sim_matrix, cmap='viridis', vmin=0, vmax=1)
    plt.colorbar(label='Similarity')
    plt.xticks(range(len(subjects)), subjects, rotation=90)
    plt.yticks(range(len(subjects)), subjects)
    plt.title('Subject Similarity Based on Top EEG Channels (Spatially Weighted)')
    plt.tight_layout()
    plt.show()

def cluster_subjects(top_channels_by_subject, ch_names, channel_dist_matrices, 
                    sigma=0.1, method='average', band_name=None, amp_factor=None):
    """
    Cluster subjects based on similarity of top channels with spatial weighting.
    
    Parameters:
    -----------
    top_channels_by_subject : dict
        Dictionary mapping subject IDs to lists of their top channels
    ch_names : list
        List of all channel names
    channel_dist_matrices : dict
        Dictionary mapping subject IDs to their channel distance matrices
    sigma : float
        Parameter controlling how quickly similarity falls off with distance
    method : str
        Linkage method for hierarchical clustering
    band_name : str, optional
        Name of frequency band (for plot title)
    amp_factor : float, optional
        Amplification factor (for plot title)
    """
    # Compute similarity matrix
    sim_matrix, subjects = compute_similarity_matrix(
        top_channels_by_subject, ch_names, channel_dist_matrices, sigma
    )
    
    # Visualize similarity matrix
    title_suffix = ""
    if band_name:
        title_suffix += f" - {band_name} band"
    if amp_factor:
        title_suffix += f" (amp factor: {amp_factor})"
        
    plt.figure(figsize=(12, 4))
    plt.imshow(sim_matrix, cmap='viridis', vmin=0, vmax=1)
    plt.colorbar(label='Similarity')
    plt.xticks(range(len(subjects)), subjects, rotation=90)
    plt.yticks(range(len(subjects)), subjects)
    plt.title(f'Subject Similarity Based on Top EEG Channels{title_suffix}')
    plt.tight_layout()
    plt.show()
    
    # Convert similarity to distance for clustering
    dist_matrix = 1 - sim_matrix
    
    # Perform hierarchical clustering
    linkage_matrix = linkage(squareform(dist_matrix), method=method)
    
    # Plot dendrogram
    plt.figure(figsize=(12, 4))
    dendrogram(
        linkage_matrix,
        labels=subjects,
        leaf_font_size=10
    )
    plt.title(f'Hierarchical Clustering of Subjects{title_suffix}')
    plt.xlabel('Distance')
    plt.tight_layout()
    plt.show()
    
    return linkage_matrix, sim_matrix, subjects

def create_distance_matrices(all_subjects_data, common_channels):
    """
    Create distance matrices for each subject based on channel coordinates.
    
    Parameters:
    -----------
    all_subjects_data : dict
        Dictionary containing subject data including info_subj for each subject
        
    Returns:
    --------
    dict : Dictionary mapping subject IDs to their channel distance matrices
    """
    channel_dist_matrices = {}
    
    for subject_id, subject_data in all_subjects_data.items():
        info = subject_data["info_subj"]
        ch_names = subject_data["ch_names"]
        
        # Extract channel coordinates
        channel_coordinates = {}
        for ch in common_channels:

            # Get the 3D coordinates (first 3 elements of loc)
            idx = ch_names.index(ch)
            channel_coordinates[ch] = info['chs'][idx]["loc"][:3]
        
        # Create array of channel locations
        channel_locations = np.array([channel_coordinates[ch] for ch in common_channels])
        
        # Compute distance matrix
        channel_dist_matrices[subject_id] = squareform(pdist(channel_locations))
    
    return channel_dist_matrices

def analyze_frequency_band(top_channels_median_all_subjects, all_subjects_data, common_channels,
                          band_name="alpha", amp_factor=2, sigma=0.1):
    """
    Analyze clustering for a specific frequency band and amplification factor.
    
    Parameters:
    -----------
    top_channels_median_all_subjects : dict
        Nested dictionary with structure [subject_id][band_name][amp_factor] = list of channels
    all_subjects_data : dict
        Dictionary containing subject data
    band_name : str
        Name of frequency band to analyze
    amp_factor : float
        Amplification factor to analyze
    sigma : float
        Parameter controlling spatial similarity weighting
    """

    top_channels_by_subject = {}
    for subject_id in top_channels_median_all_subjects:
        top_channels_by_subject[subject_id] = top_channels_median_all_subjects[subject_id][band_name][amp_factor]
    

    channel_dist_matrices = create_distance_matrices(all_subjects_data, common_channels)
    
    # Perform clustering
    linkage_matrix, sim_matrix, subjects = cluster_subjects(
        top_channels_by_subject, 
        common_channels, 
        channel_dist_matrices, 
        sigma=sigma,
        method='average',
        band_name=band_name,
        amp_factor=amp_factor
    )
    distance_matrix = 1 - sim_matrix
    dir = "distance_matrices"
    os.makedirs(dir, exist_ok=True)
    np.save(f"{dir}/distance_matrix_power_{band_name}_{amp_factor}.npy", distance_matrix)
    
    return linkage_matrix, sim_matrix, subjects

def analyze_all_bands(top_channels_median_all_subjects, all_subjects_data, common_channels, 
                     amplification_factors=[0.5, 0.75, 1.5, 2, 3], 
                     freq_bands=["theta", "delta", "alpha", "beta", "gamma"],
                     sigma=0.1):
    """
    Analyze clustering across all frequency bands and amplification factors.
    
    Parameters:
    -----------
    top_channels_median_all_subjects : dict
        Nested dictionary with structure [subject_id][band_name][amp_factor] = list of channels
    all_subjects_data : dict
        Dictionary containing subject data
    amplification_factors : list
        List of amplification factors to analyze
    freq_bands : list
        List of frequency bands to analyze
    sigma : float
        Parameter controlling spatial similarity weighting
    
    Returns:
    --------
    dict : Results for each band and amplification factor
    """
    results = {}
    
    for band_name in freq_bands:
        results[band_name] = {}
        
        for amp_factor in amplification_factors:
            print(f"Analyzing {band_name} band with amplification factor {amp_factor}...")
            
            linkage_matrix, sim_matrix, subjects = analyze_frequency_band(
                top_channels_median_all_subjects,
                all_subjects_data,
                common_channels,
                band_name,
                amp_factor,
                sigma=sigma
            )
            
            results[band_name][amp_factor] = {
                'linkage_matrix': linkage_matrix,
                'sim_matrix': sim_matrix,
                'subjects': subjects
            }
    
    return results

def plot_cluster_consistency(results, freq_bands, amplification_factors, method='adjusted_rand_score'):
    """
    Plot the consistency of clusters across frequency bands and amplification factors.
    
    Parameters:
    -----------
    results : dict
        Results from analyze_all_bands function
    freq_bands : list
        List of frequency bands
    amplification_factors : list
        List of amplification factors
    method : str
        Method to compute cluster consistency ('adjusted_rand_score' or 'adjusted_mutual_info_score')
    """
    try:
        from sklearn.metrics import adjusted_rand_score, adjusted_mutual_info_score
    except ImportError:
        %pip install scikit-learn
        from sklearn.metrics import adjusted_rand_score, adjusted_mutual_info_score
    
    # Choose scoring method
    if method == 'adjusted_rand_score':
        score_func = adjusted_rand_score
    else:
        score_func = adjusted_mutual_info_score
    
    n_bands = len(freq_bands)
    n_factors = len(amplification_factors)
    
    # Create matrices to store pairwise comparisons
    band_comparison = np.zeros((n_bands, n_bands))
    factor_comparison = np.zeros((n_factors, n_factors))
    
    # Extract cluster labels from hierarchical clustering
    def get_cluster_labels(linkage_matrix, n_clusters=3):
        """Extract cluster labels from linkage matrix"""
        from scipy.cluster.hierarchy import fcluster
        return fcluster(linkage_matrix, n_clusters, criterion='maxclust')
    
    # Compare across frequency bands (using the middle amplification factor)
    mid_factor = amplification_factors[int((len(amplification_factors)//2))]
    for i, band1 in enumerate(freq_bands):
        for j, band2 in enumerate(freq_bands):
            if i <= j:  # Only compute upper triangle (including diagonal)
                labels1 = get_cluster_labels(results[band1][mid_factor]['linkage_matrix'])
                labels2 = get_cluster_labels(results[band2][mid_factor]['linkage_matrix'])
                score = score_func(labels1, labels2)
                band_comparison[i, j] = score
                band_comparison[j, i] = score  # Mirror the matrix
    
    # Compare across amplification factors (using the middle frequency band)
    mid_band = freq_bands[len(freq_bands)//2]
    for i, factor1 in enumerate(amplification_factors):
        for j, factor2 in enumerate(amplification_factors):
            if i <= j:  # Only compute upper triangle (including diagonal)
                labels1 = get_cluster_labels(results[mid_band][factor1]['linkage_matrix'])
                labels2 = get_cluster_labels(results[mid_band][factor2]['linkage_matrix'])
                score = score_func(labels1, labels2)
                factor_comparison[i, j] = score
                factor_comparison[j, i] = score  # Mirror the matrix
    
    # Plot frequency band comparison
    plt.figure(figsize=(10, 8))
    plt.imshow(band_comparison, cmap='viridis', vmin=0, vmax=1)
    plt.colorbar(label='Cluster Similarity')
    plt.xticks(range(n_bands), freq_bands)
    plt.yticks(range(n_bands), freq_bands)
    plt.title(f'Cluster Consistency Across Frequency Bands ({method})')
    plt.tight_layout()
    plt.show()
    
    # Plot amplification factor comparison
    plt.figure(figsize=(10, 8))
    plt.imshow(factor_comparison, cmap='viridis', vmin=0, vmax=1)
    plt.colorbar(label='Cluster Similarity')
    plt.xticks(range(n_factors), amplification_factors)
    plt.yticks(range(n_factors), amplification_factors)
    plt.title(f'Cluster Consistency Across Amplification Factors ({method})')
    plt.tight_layout()
    plt.show()
    
    return band_comparison, factor_comparison

In [ ]:
def get_common_channels(all_subjects_data):
    # Get the first subject's channel names as a set
    first_subject = list(all_subjects_data.keys())[0]
    common_channels = set(all_subjects_data[first_subject]["ch_names"])
    
    # Intersect with each other subject's channels
    for subject_id in all_subjects_data.keys():
        subject_channels = set(all_subjects_data[subject_id]["ch_names"])
        common_channels = common_channels.intersection(subject_channels)
    
    # Convert back to list and sort for consistency
    common_channels = sorted(list(common_channels))
    
    # Print summary
    print(f"Found {len(common_channels)} common channels across {len(all_subjects_data)} subjects")
    print("Common channels:", common_channels)
    
    return common_channels

In [ ]:
common_channels = get_common_channels(all_subjects_data)

In [ ]:
channel_dist_matrices = create_distance_matrices(all_subjects_data, common_channels)

analyze which channel names are in all subject datas.
remove all other channels from the subject data for all subjects

need to find out which indices they correspond to in different subjects
only compute results based these channels?
Potentially good enough to remove all channels from the top 10 that are not in the common channels

In [ ]:
def filter_top_channels_to_common(top_channels_median_all_subjects, common_channels):
    """
    Filter top channels for all subjects to only include channels that are common across all subjects.
    
    Parameters:
    -----------
    top_channels_median_all_subjects : dict
        Dictionary containing top channels for each subject
    common_channels : list
        List of channel names that are common across all subjects
        
    Returns:
    --------
    dict : Filtered dictionary with only common channels
    """
    filtered_top_channels = {}
    
    # Iterate through each subject
    for subject_id in top_channels_median_all_subjects:
        filtered_top_channels[subject_id] = {}
        
        # Iterate through each frequency band
        for band_name in top_channels_median_all_subjects[subject_id]:
            filtered_top_channels[subject_id][band_name] = {}
            
            # Iterate through each amplification factor
            for factor in top_channels_median_all_subjects[subject_id][band_name]:
                # Get original top channels for this combination
                original_channels = top_channels_median_all_subjects[subject_id][band_name][factor]
                
                # Filter to only include common channels
                filtered_channels = [ch for ch in original_channels if ch in common_channels]
                
                filtered_top_channels[subject_id][band_name][factor] = filtered_channels
    
    return filtered_top_channels

# Filter the top channels
filtered_top_channels_median_all_subjects = filter_top_channels_to_common(
    top_channels_median_all_subjects, 
    common_channels
)

In [ ]:
def get_cluster_labels(linkage_matrix, n_clusters=4):
    """Extract cluster labels from linkage matrix"""

    return fcluster(linkage_matrix, n_clusters, criterion='maxclust')

In [ ]:
def plot_cluster_topoplots(linkage_matrix, subjects, all_subjects_data, common_channels, band_name="gamma", n_clusters=4, amp_factor=2, pairwise=True, ax=None):
    """
    Plot topographic maps of cluster centroids for each cluster.
    
    Parameters:
    -----------
    linkage_matrix : np.array
        Linkage matrix from hierarchical clustering
    subjects : list
        List of subject IDs
    all_subjects_data : dict
        Dictionary containing subject data
    common_channels : list
        List of common channel names
    n_clusters : int
        Number of clusters to extract
    ax : matplotlib.axes.Axes or array of Axes, optional
        The axes to plot on. If None, creates new figure.
    """
    # Extract cluster labels
    cluster_labels = get_cluster_labels(linkage_matrix, n_clusters)
    np.save(f"cluster_labels_{band_name}_{amp_factor}_n_clusters_{n_clusters}_power.npy", cluster_labels)
    
    # Create figure if ax is None
    if ax is None:
        fig, axs = plt.subplots(1, n_clusters, figsize=(15, 5))
    else:
        # If ax is a single axis, make it a list for consistency
        if isinstance(ax, plt.Axes):
            axs = [ax]
        else:
            axs = ax
        fig = axs[0].figure
        
    ch_names = all_subjects_data[1]["ch_names"]
    
    # Iterate through each cluster
    for cluster_id in range(1, n_clusters + 1):
        # Check if we have enough axes to plot on
  
        # Find subjects in this cluster
        cluster_subjects = [subj for subj, label in zip(subjects, cluster_labels) if label == cluster_id]
        
        # Average the top channels for these subjects
        all_agree_channels_cluster = np.zeros(len(ch_names))
        if pairwise:
            for subject_id1, subject_id2 in itertools.combinations(cluster_subjects, 2):
                pair_agreement = np.intersect1d(top_channels_median_all_subjects[subject_id1][band_name][amp_factor], 
                                              top_channels_median_all_subjects[subject_id2][band_name][amp_factor])
                pair_agreement = [ch for ch in pair_agreement if ch in common_channels]
                for ch in pair_agreement:
                    all_agree_channels_cluster[ch_names.index(ch)] += 1
        else:
            for subject_id in cluster_subjects:
                for ch in filtered_top_channels_median_all_subjects[subject_id][band_name][amp_factor]:
                    if ch in common_channels:
                        all_agree_channels_cluster[ch_names.index(ch)] += 1

        # Plot topomap
        if n_clusters > 1:
            current_ax = axs[cluster_id - 1]
        else:
            current_ax = axs
        mne.viz.plot_topomap(all_agree_channels_cluster, all_subjects_data[1]["info_subj"], 
                           show=False, names=ch_names, axes=current_ax)
      
        current_ax.set_title(f'Cluster {cluster_id}, N: {len(cluster_subjects)}', fontsize=18)
    
    if ax is None:
        plt.suptitle('Average topomap of clusters', fontsize=20)
        plt.tight_layout()
        fig.savefig(f"cluster_topo_{band_name}_{amp_factor}_power_n_clusters_{n_clusters}_pairwise_{pairwise}.png")
        plt.show()
        
    return axs

# across amplification factors

In [ ]:
for freq_band in freq_bands.keys():
    for amp_factor in amplification_factors:
        linkage_matrix, sim_matrix, subjects = analyze_frequency_band(filtered_top_channels_median_all_subjects, all_subjects_data, common_channels, band_name=freq_band, amp_factor=amp_factor, sigma=0.0025)
        plot_cluster_topoplots(linkage_matrix, subjects, all_subjects_data, common_channels, band_name=freq_band, n_clusters=4, amp_factor=amp_factor)
        plot_cluster_topoplots(linkage_matrix, subjects, all_subjects_data, common_channels, band_name=freq_band, n_clusters=4, amp_factor=amp_factor, pairwise=False)

while positions seem correct, the names of the positions are wrong right now

# cluster for different frequency bands but single factor

In [ ]:
import numpy as np
from scipy.cluster.hierarchy import linkage, fcluster
from sklearn.metrics import silhouette_score, davies_bouldin_score

# Assuming `distance_matrix` is your precomputed distance matrix
def evaluate_n_clusters(distance_matrix, method='average', max_k=10):

# Store results
    results = {}


    Z = linkage(distance_matrix, method=method)
    silhouette_scores = []
    db_scores = []
    within_cluster_dists = []
    
    for k in range(2, max_k + 1):
        labels = fcluster(Z, k, criterion='maxclust')
        silhouette = silhouette_score(distance_matrix, labels, metric='precomputed')
        db = davies_bouldin_score(distance_matrix, labels)
        
        # Compute within-cluster sum of distances
        within_dist = 0
        for cluster in np.unique(labels):
            indices = np.where(labels == cluster)[0]
            if len(indices) > 1:
                within_dist += np.sum(distance_matrix[np.ix_(indices, indices)])
        within_cluster_dists.append(within_dist)
        
        silhouette_scores.append(silhouette)
        db_scores.append(db)
    
    results[method] = {
        'silhouette': silhouette_scores,
        'davies_bouldin': db_scores,
        'within_cluster_dist': within_cluster_dists
    }

    return results


In [ ]:
evaluate_n_clusters(1-sim_matrix, method='average', max_k=6)

In [ ]:
linkage_matrix, sim_matrix, subjects = analyze_frequency_band(
    filtered_top_channels_median_all_subjects,
    all_subjects_data,
    common_channels,
    band_name="theta",  # Choose your frequency band
    amp_factor=2,       # Choose your amplification factor
    sigma=0.0025 # Adjust based on your spatial scale
)

In [ ]:
plot_cluster_topoplots(linkage_matrix, subjects, all_subjects_data, common_channels, n_clusters=1, band_name="theta",amp_factor=2)
plot_cluster_topoplots(linkage_matrix, subjects, all_subjects_data, common_channels, n_clusters=1, band_name="theta",amp_factor=2, pairwise=False)

In [ ]:
linkage_matrix, sim_matrix, subjects = analyze_frequency_band(
    filtered_top_channels_median_all_subjects,
    all_subjects_data,
    common_channels,
    band_name="delta",  # Choose your frequency band
    amp_factor=2,       # Choose your amplification factor
    sigma=0.0025 # Adjust based on your spatial scale
)

In [ ]:
evaluate_n_clusters(1-sim_matrix, method='average', max_k=6)

In [ ]:
plot_cluster_topoplots(linkage_matrix, subjects, all_subjects_data, common_channels, n_clusters=2, band_name="delta",amp_factor=2)
plot_cluster_topoplots(linkage_matrix, subjects, all_subjects_data, common_channels, n_clusters=2, band_name="delta",amp_factor=2, pairwise=False)

In [ ]:
plot_cluster_topoplots(linkage_matrix, subjects, all_subjects_data, common_channels, n_clusters=3, band_name="delta",amp_factor=2)
plot_cluster_topoplots(linkage_matrix, subjects, all_subjects_data, common_channels, n_clusters=3, band_name="delta",amp_factor=2, pairwise=False)

In [ ]:
plot_cluster_topoplots(linkage_matrix, subjects, all_subjects_data, common_channels, n_clusters=4, band_name="delta",amp_factor=2)
plot_cluster_topoplots(linkage_matrix, subjects, all_subjects_data, common_channels, n_clusters=4, band_name="delta",amp_factor=2, pairwise=False)

In [ ]:
plot_cluster_topoplots(linkage_matrix, subjects, all_subjects_data, common_channels, n_clusters=3, band_name="delta",amp_factor=2)
plot_cluster_topoplots(linkage_matrix, subjects, all_subjects_data, common_channels, n_clusters=3, band_name="delta",amp_factor=2, pairwise=False)

In [ ]:
linkage_matrix, sim_matrix, subjects = analyze_frequency_band(
    filtered_top_channels_median_all_subjects,
    all_subjects_data,
    common_channels,
    band_name="alpha",  # Choose your frequency band
    amp_factor=2,       # Choose your amplification factor
    sigma=0.0025 # Adjust based on your spatial scale
)

In [ ]:
evaluate_n_clusters(1-sim_matrix, method='average', max_k=6)

In [ ]:
plot_cluster_topoplots(linkage_matrix, subjects, all_subjects_data, common_channels, n_clusters=1, band_name="alpha",amp_factor=2)
plot_cluster_topoplots(linkage_matrix, subjects, all_subjects_data, common_channels, n_clusters=1, band_name="alpha",amp_factor=2, pairwise=False)

In [ ]:
plot_cluster_topoplots(linkage_matrix, subjects, all_subjects_data, common_channels, n_clusters=2, band_name="alpha",amp_factor=2)
plot_cluster_topoplots(linkage_matrix, subjects, all_subjects_data, common_channels, n_clusters=2, band_name="alpha",amp_factor=2, pairwise=False)

In [ ]:
plot_cluster_topoplots(linkage_matrix, subjects, all_subjects_data, common_channels, n_clusters=3, band_name="alpha",amp_factor=2)
plot_cluster_topoplots(linkage_matrix, subjects, all_subjects_data, common_channels, n_clusters=3, band_name="alpha",amp_factor=2, pairwise=False)

In [ ]:
fig, axs = plt.subplots(nrows=3, ncols=4, figsize=(20, 12))
plt.subplots_adjust(wspace=-0.6)
bands = ["theta", "delta", "alpha"]
fig.suptitle("Average topomap of power clusters in theta, delta, alpha bands", fontsize=20)
for i, band in enumerate(["theta", "delta", "alpha"]):
    linkage_matrix, sim_matrix, subjects = analyze_frequency_band(
        filtered_top_channels_median_all_subjects,
        all_subjects_data,
        common_channels,
        band_name=band, 
        amp_factor=2,       
        sigma=0.0025 
    )
    plot_cluster_topoplots(linkage_matrix, subjects, all_subjects_data, common_channels, n_clusters=4, band_name=band, amp_factor=2, ax=axs[i])
#fig.suptitle("Average topomap of clusters in theta, delta, alpha bands", fontsize=20)
fig.savefig("Topomap_theta_delta_alpha_power.png")


In [ ]:
linkage_matrix, sim_matrix, subjects = analyze_frequency_band(
    filtered_top_channels_median_all_subjects,
    all_subjects_data,
    common_channels,
    band_name="beta",  # Choose your frequency band
    amp_factor=2,       # Choose your amplification factor
    sigma=0.0025 # Adjust based on your spatial scale
)

In [ ]:
evaluate_n_clusters(1-sim_matrix, method='average', max_k=6)

In [ ]:
plot_cluster_topoplots(linkage_matrix, subjects, all_subjects_data, common_channels, n_clusters=1, band_name="beta",amp_factor=2)
plot_cluster_topoplots(linkage_matrix, subjects, all_subjects_data, common_channels, n_clusters=1, band_name="beta",amp_factor=2, pairwise=False)

In [ ]:
plot_cluster_topoplots(linkage_matrix, subjects, all_subjects_data, common_channels, n_clusters=2, band_name="beta",amp_factor=2)
plot_cluster_topoplots(linkage_matrix, subjects, all_subjects_data, common_channels, n_clusters=2, band_name="beta",amp_factor=2, pairwise=False)

In [ ]:
plot_cluster_topoplots(linkage_matrix, subjects, all_subjects_data, common_channels, n_clusters=3, band_name="beta",amp_factor=2)
plot_cluster_topoplots(linkage_matrix, subjects, all_subjects_data, common_channels, n_clusters=3, band_name="beta",amp_factor=2, pairwise=False)

In [ ]:
linkage_matrix, sim_matrix, subjects = analyze_frequency_band(
    filtered_top_channels_median_all_subjects,
    all_subjects_data,
    common_channels,
    band_name="gamma",  # Choose your frequency band
    amp_factor=2,       # Choose your amplification factor
    sigma=0.0025 # Adjust based on your spatial scale
)

In [ ]:
evaluate_n_clusters(1-sim_matrix, method='average', max_k=6)

In [ ]:
plot_cluster_topoplots(linkage_matrix, subjects, all_subjects_data, common_channels, n_clusters=2, band_name="gamma",amp_factor=2)
plot_cluster_topoplots(linkage_matrix, subjects, all_subjects_data, common_channels, n_clusters=2, band_name="gamma",amp_factor=2, pairwise=False)

In [ ]:
plot_cluster_topoplots(linkage_matrix, subjects, all_subjects_data, common_channels, n_clusters=3, band_name="gamma",amp_factor=2)
plot_cluster_topoplots(linkage_matrix, subjects, all_subjects_data, common_channels, n_clusters=3, band_name="gamma",amp_factor=2, pairwise=False)

In [ ]:
plot_cluster_topoplots(linkage_matrix, subjects, all_subjects_data, common_channels, n_clusters=4, band_name="gamma",amp_factor=2)
plot_cluster_topoplots(linkage_matrix, subjects, all_subjects_data, common_channels, n_clusters=4, band_name="gamma",amp_factor=2, pairwise=False)

In [ ]:
fig, axs = plt.subplots(nrows=2, ncols=4, figsize=(20, 8))
plt.subplots_adjust(wspace=-0.6)
bands = ["beta", "gamma"]
fig.suptitle("Average topomap of power clusters in beta, gamma bands", fontsize=20)
for i, band in enumerate(bands):
    linkage_matrix, sim_matrix, subjects = analyze_frequency_band(
        filtered_top_channels_median_all_subjects,
        all_subjects_data,
        common_channels,
        band_name=band, 
        amp_factor=2,       
        sigma=0.0025 
    )
    plot_cluster_topoplots(linkage_matrix, subjects, all_subjects_data, common_channels, n_clusters=4, band_name=band, amp_factor=2, ax=axs[i])
#fig.suptitle("Average topomap of clusters in theta, delta, alpha bands", fontsize=20)
fig.savefig("Topomap_beta_gamma_power.png")

plotting mean topoplot for each cluster in each frequency band would probably be nice to get an intuition into clustering

maybe setting would need to be adapted for this

results = analyze_all_bands(
    filtered_top_channels_median_all_subjects,
    all_subjects_data,
    common_channels,
    amplification_factors=[0.5, 0.75, 1.5, 2, 3],
    freq_bands=["theta", "delta", "alpha", "beta", "gamma"],
    sigma=0.0025
)

# cluster based on rank correlations

In [ ]:
def get_common_channels(all_subjects_data):
    # Get the first subject's channel names as a set
    first_subject = list(all_subjects_data.keys())[0]
    common_channels = set(all_subjects_data[first_subject]["ch_names"])
    
    # Intersect with each other subject's channels
    for subject_id in all_subjects_data.keys():
        subject_channels = set(all_subjects_data[subject_id]["ch_names"])
        common_channels = common_channels.intersection(subject_channels)
    
    # Convert back to list and sort for consistency
    common_channels = sorted(list(common_channels))
    
    # Print summary
    print(f"Found {len(common_channels)} common channels across {len(all_subjects_data)} subjects")
    print("Common channels:", common_channels)
    
    return common_channels
common_channels = get_common_channels(all_subjects_data)

In [ ]:
import scipy
def calculate_pairwise_correlations(median_diff1, median_diff2, freq_bands, amplification_factors, common_channels):

    correlation_results = {}
    correlation_results_mean = {}

    for band_name in freq_bands.keys():
        correlation_results[band_name] = {}
        for factor in amplification_factors:
            # Extract the top 10 channels for both subjects
            data1 =median_diff1[band_name][factor]
            data2 = median_diff2[band_name][factor]
            # Extract values for common channels only
            common_channel_values1 = np.array([data1[ch] for ch in common_channels])
            common_channel_values2 = np.array([data2[ch] for ch in common_channels])
            
            # Calculate correlations
            pearson_corr = np.corrcoef(common_channel_values1, common_channel_values2)[0,1]
            spearman_corr = scipy.stats.spearmanr(common_channel_values1, common_channel_values2).correlation
            
            # Store results
            correlation_results[band_name][factor] = {
                'pearson': pearson_corr,
                'spearman': spearman_corr
            }
            # Calculate agreement
        # compute the mean correlation across factors
        correlation_results_mean[band_name] = {
            'pearson': np.mean([correlation_results[band_name][factor]['pearson'] for factor in amplification_factors]),
            'spearman': np.mean([correlation_results[band_name][factor]['spearman'] for factor in amplification_factors])
        }

    return correlation_results, correlation_results_mean

In [ ]:
def plot_agreement_matrix_correlations(correlation_results, sub1_index, sub2_index):
    # plot a matrix with separate row for each frequency band and separate column for each amplification factor
    bands = list(correlation_results.keys())
    factors = list(correlation_results[bands[0]].keys())
    correlation_matrix_pearson = np.zeros((len(bands), len(factors)))
    correlation_matrix_spearman = np.zeros((len(bands), len(factors)))
    
    fig,axs = plt.subplots(1, 2, figsize=(15, 5), sharex=True, sharey=True)
    fig.subplots_adjust(wspace=-0.2)  # Reduce horizontal space between subplots
    fig.suptitle(f'Correlation between subjects {sub1_index} and {sub2_index}')
    
    # Fill correlation matrices
    for i, band in enumerate(bands):
        for j, factor in enumerate(factors):
            correlation_matrix_pearson[i, j] = correlation_results[band][factor]['pearson']
            correlation_matrix_spearman[i, j] = correlation_results[band][factor]['spearman']

    
    im_pearson = axs[0].matshow(correlation_matrix_pearson, cmap='viridis', vmin=0, vmax=1)
    im_spearman = axs[1].matshow(correlation_matrix_spearman, cmap='viridis', vmin=0, vmax=1)

    # Add single colorbar for both plots
    cbar = fig.colorbar(im_spearman, ax=axs.ravel().tolist(), location='bottom', shrink=0.35)
    cbar.set_label('Correlation')

    # Set axis labels
    for ax in axs:
        ax.set_xticks(np.arange(len(factors)))
        ax.set_yticks(np.arange(len(bands)))
        ax.set_xticklabels(factors)
        ax.set_yticklabels(bands)
        ax.set_xlabel('Amplification factors')
        ax.set_ylabel('Frequency Bands')
        plt.setp(ax.get_xticklabels(), rotation=45, ha="left", rotation_mode="anchor")
    
    axs[0].set_title('Pearson correlation')
    axs[1].set_title('Spearman correlation')

    # Add text annotations
    for i in range(len(bands)):
        for j in range(len(factors)):
            axs[0].text(j, i, f"{correlation_matrix_pearson[i, j]:.2f}", 
                       ha="center", va="center", color="white")
            axs[1].text(j, i, f"{correlation_matrix_spearman[i, j]:.2f}", 
                       ha="center", va="center", color="white")


In [ ]:
cfg = load_config()
rank_correlations_mean_all_pairs = {}
for subject_index1, subject_index2 in itertools.combinations(cfg.dataset.test_subject_indices,2):
    fig = plt.figure(layout='constrained', figsize=(15, 4))
    subfigs = fig.subfigures(1, 2, wspace=0.07)
    correlation_results, correlation_results_mean = calculate_pairwise_correlations(median_diff_per_channel_all_subjects[subject_index1], median_diff_per_channel_all_subjects[subject_index2], freq_bands, amplification_factors, common_channels)
    rank_correlations_mean_all_pairs[(subject_index1, subject_index2)] = correlation_results_mean
    #plot_agreement_matrix_correlations(correlation_results, subject_index1, subject_index2)
    

In [ ]:
def plot_agreement_matrix_correlations_all_subjects(mean_correlations_all_subject_pairs, freq_band="alpha", take_abs=False, plot=True):
    # for a given frequency band and amplification factor, plot the correlation between all subjects
    # subjcet indicies are on rows and columns
    # the value in each cell is the mean correlation across all factors
    #
    correlation_matrix_pearson = np.ones((len(cfg.dataset.test_subject_indices), len(cfg.dataset.test_subject_indices)))
    correlation_matrix_spearman = np.ones((len(cfg.dataset.test_subject_indices), len(cfg.dataset.test_subject_indices)))
    subject_indices = list(cfg.dataset.test_subject_indices)                             
    for subj1,subj2 in itertools.combinations(subject_indices,2):
        if take_abs:
            correlation_matrix_pearson[subject_indices.index(subj1),subject_indices.index(subj2)] = np.abs(mean_correlations_all_subject_pairs[(subj1,subj2)][freq_band]['pearson'])
            correlation_matrix_pearson[subject_indices.index(subj2),subject_indices.index(subj1)] = np.abs(mean_correlations_all_subject_pairs[(subj1,subj2)][freq_band]['pearson'])
            correlation_matrix_spearman[subject_indices.index(subj1),subject_indices.index(subj2)] = np.abs(mean_correlations_all_subject_pairs[(subj1,subj2)][freq_band]['spearman'])
            correlation_matrix_spearman[subject_indices.index(subj2),subject_indices.index(subj1)] = np.abs(mean_correlations_all_subject_pairs[(subj1,subj2)][freq_band]['spearman'])
        else:
            correlation_matrix_pearson[subject_indices.index(subj1),subject_indices.index(subj2)] = mean_correlations_all_subject_pairs[(subj1,subj2)][freq_band]['pearson']
            correlation_matrix_pearson[subject_indices.index(subj2),subject_indices.index(subj1)] = mean_correlations_all_subject_pairs[(subj1,subj2)][freq_band]['pearson']
            correlation_matrix_spearman[subject_indices.index(subj1),subject_indices.index(subj2)] = mean_correlations_all_subject_pairs[(subj1,subj2)][freq_band]['spearman']
            correlation_matrix_spearman[subject_indices.index(subj2),subject_indices.index(subj1)] = mean_correlations_all_subject_pairs[(subj1,subj2)][freq_band]['spearman']
    
        # use the correlations matrices as distance metrics:
    distance_matrix_pearson = 1 - correlation_matrix_pearson
    distance_matrix_spearman = 1 - correlation_matrix_spearman
    os.makedirs("distance_matrices", exist_ok=True)
    np.save(f"distance_matrices/power_correlation_matrix_pearson_abs_freq_band_{freq_band}.npy", distance_matrix_pearson)
    np.save(f"distance_matrices/power_correlation_matrix_spearman_abs_freq_band_{freq_band}.npy", distance_matrix_spearman)

    if plot:
        fig,axs = plt.subplots(nrows=2, ncols=1, figsize=(25, 25))

        axs[0].matshow(correlation_matrix_pearson, cmap='viridis', vmin=-1, vmax=1)
        axs[1].matshow(correlation_matrix_spearman, cmap='viridis', vmin=-1, vmax=1)

        # Set axis labels
        for ax in axs:
            ax.set_xticks(np.arange(len(subject_indices)))
            ax.set_yticks(np.arange(len(subject_indices)))
            ax.set_xticklabels(subject_indices)
            ax.set_yticklabels(subject_indices)
            ax.set_xlabel('Subject indices')
            ax.set_ylabel('Subject indices')
            plt.setp(ax.get_xticklabels(), rotation=45, ha="left", rotation_mode="anchor")
        
        axs[0].set_title('Pearson correlation')
        axs[1].set_title('Spearman correlation')

        # Add text annotations
        for i in range(len(subject_indices)):
            for j in range(len(subject_indices)):
                axs[0].text(j, i, f"{correlation_matrix_pearson[i, j]:.2f}", 
                           ha="center", va="center", color="white")
                axs[1].text(j, i, f"{correlation_matrix_spearman[i, j]:.2f}", 
                           ha="center", va="center", color="white")
        
        return fig, correlation_matrix_pearson, correlation_matrix_spearman
    
    return None, correlation_matrix_pearson, correlation_matrix_spearman


                                           

In [ ]:
plot_agreement_matrix_correlations_all_subjects(rank_correlations_mean_all_pairs, freq_band="theta", take_abs=False)

In [ ]:
plot_agreement_matrix_correlations_all_subjects(rank_correlations_mean_all_pairs, freq_band="delta", take_abs=False)

In [ ]:
plot_agreement_matrix_correlations_all_subjects(rank_correlations_mean_all_pairs, freq_band="alpha", take_abs=False)

In [ ]:
plot_agreement_matrix_correlations_all_subjects(rank_correlations_mean_all_pairs, freq_band="beta", take_abs=False)

In [ ]:
plot_agreement_matrix_correlations_all_subjects(rank_correlations_mean_all_pairs, freq_band="gamma", take_abs=False)

## dont take absolute value in median differences

In [ ]:
def plot_agreement_matrix_correlations_all_subjects(mean_correlations_all_subject_pairs, freq_band="alpha", take_abs=False, plot=True):
    # for a given frequency band and amplification factor, plot the correlation between all subjects
    # subjcet indicies are on rows and columns
    # the value in each cell is the mean correlation across all factors
    #
    correlation_matrix_pearson = np.ones((len(cfg.dataset.test_subject_indices), len(cfg.dataset.test_subject_indices)))
    correlation_matrix_spearman = np.ones((len(cfg.dataset.test_subject_indices), len(cfg.dataset.test_subject_indices)))
    subject_indices = list(cfg.dataset.test_subject_indices)                             
    for subj1,subj2 in itertools.combinations(subject_indices,2):
        if take_abs:
            correlation_matrix_pearson[subject_indices.index(subj1),subject_indices.index(subj2)] = np.abs(mean_correlations_all_subject_pairs[(subj1,subj2)][freq_band]['pearson'])
            correlation_matrix_pearson[subject_indices.index(subj2),subject_indices.index(subj1)] = np.abs(mean_correlations_all_subject_pairs[(subj1,subj2)][freq_band]['pearson'])
            correlation_matrix_spearman[subject_indices.index(subj1),subject_indices.index(subj2)] = np.abs(mean_correlations_all_subject_pairs[(subj1,subj2)][freq_band]['spearman'])
            correlation_matrix_spearman[subject_indices.index(subj2),subject_indices.index(subj1)] = np.abs(mean_correlations_all_subject_pairs[(subj1,subj2)][freq_band]['spearman'])
        else:
            correlation_matrix_pearson[subject_indices.index(subj1),subject_indices.index(subj2)] = mean_correlations_all_subject_pairs[(subj1,subj2)][freq_band]['pearson']
            correlation_matrix_pearson[subject_indices.index(subj2),subject_indices.index(subj1)] = mean_correlations_all_subject_pairs[(subj1,subj2)][freq_band]['pearson']
            correlation_matrix_spearman[subject_indices.index(subj1),subject_indices.index(subj2)] = mean_correlations_all_subject_pairs[(subj1,subj2)][freq_band]['spearman']
            correlation_matrix_spearman[subject_indices.index(subj2),subject_indices.index(subj1)] = mean_correlations_all_subject_pairs[(subj1,subj2)][freq_band]['spearman']

        # use the correlations matrices as distance metrics:
    distance_matrix_pearson = 1 - correlation_matrix_pearson
    distance_matrix_spearman = 1 - correlation_matrix_spearman
    os.makedirs("distance_matrices", exist_ok=True)
    np.save(f"distance_matrices/power_correlation_matrix_pearson_freq_band_{freq_band}.npy", distance_matrix_pearson)
    np.save(f"distance_matrices/power_correlation_matrix_spearman_freq_band_{freq_band}.npy", distance_matrix_spearman)

    if plot:
        fig,axs = plt.subplots(nrows=2, ncols=1, figsize=(25, 25))

        axs[0].matshow(correlation_matrix_pearson, cmap='viridis', vmin=-1, vmax=1)
        axs[1].matshow(correlation_matrix_spearman, cmap='viridis', vmin=-1, vmax=1)

        # Set axis labels
        for ax in axs:
            ax.set_xticks(np.arange(len(subject_indices)))
            ax.set_yticks(np.arange(len(subject_indices)))
            ax.set_xticklabels(subject_indices)
            ax.set_yticklabels(subject_indices)
            ax.set_xlabel('Subject indices')
            ax.set_ylabel('Subject indices')
            plt.setp(ax.get_xticklabels(), rotation=45, ha="left", rotation_mode="anchor")
        
        axs[0].set_title('Pearson correlation')
        axs[1].set_title('Spearman correlation')

        # Add text annotations
        for i in range(len(subject_indices)):
            for j in range(len(subject_indices)):
                axs[0].text(j, i, f"{correlation_matrix_pearson[i, j]:.2f}", 
                           ha="center", va="center", color="white")
                axs[1].text(j, i, f"{correlation_matrix_spearman[i, j]:.2f}", 
                           ha="center", va="center", color="white")
        
        return fig, correlation_matrix_pearson, correlation_matrix_spearman
    
    return None, correlation_matrix_pearson, correlation_matrix_spearman


                                           

In [ ]:
median_diff_per_channel_all_subjects = median_difference_all_subjects(all_subjects_data, take_abs=False)

In [ ]:
cfg = load_config()
rank_correlations_mean_all_pairs = {}
for subject_index1, subject_index2 in itertools.combinations(cfg.dataset.test_subject_indices,2):
    fig = plt.figure(layout='constrained', figsize=(15, 4))
    subfigs = fig.subfigures(1, 2, wspace=0.07)
    correlation_results, correlation_results_mean = calculate_pairwise_correlations(median_diff_per_channel_all_subjects[subject_index1], median_diff_per_channel_all_subjects[subject_index2], freq_bands, amplification_factors, common_channels)
    rank_correlations_mean_all_pairs[(subject_index1, subject_index2)] = correlation_results_mean
    #plot_agreement_matrix_correlations(correlation_results, subject_index1, subject_index2)
    

In [ ]:
plot_agreement_matrix_correlations_all_subjects(rank_correlations_mean_all_pairs, freq_band="delta", take_abs=False, plot=False)
plot_agreement_matrix_correlations_all_subjects(rank_correlations_mean_all_pairs, freq_band="theta", take_abs=False, plot=False)
plot_agreement_matrix_correlations_all_subjects(rank_correlations_mean_all_pairs, freq_band="alpha", take_abs=False, plot=False)
plot_agreement_matrix_correlations_all_subjects(rank_correlations_mean_all_pairs, freq_band="beta", take_abs=False, plot=False)
plot_agreement_matrix_correlations_all_subjects(rank_correlations_mean_all_pairs, freq_band="gamma", take_abs=False, plot=False)